# Analyse Exploratoire de Données (EDA) - RetailRocket

## Dataset E-commerce comportemental

Ce notebook présente une analyse exploratoire complète du dataset RetailRocket, qui contient les interactions utilisateurs sur un site e-commerce.

### Objectifs de l'EDA :
1. Comprendre la structure des données
2. Identifier les variables clés
3. Détecter les patterns et anomalies
4. Préparer les données pour l'analyse métier

---

## 1. Configuration et Chargement des Données

In [ ]:
import pandas as pd
import numpy as np

# Librairies de visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Utilitaires
from datetime import datetime
import warnings
import os

# Configuration des warnings
warnings.filterwarnings('ignore')

# Configuration du style de visualisation
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Configuration pandas pour un meilleur affichage
pd.set_option('display.max_columns', None)      # Afficher toutes les colonnes
pd.set_option('display.max_rows', 100)          # Limite de lignes affichées
pd.set_option('display.float_format', '{:.2f}'.format)  # Format des décimales
pd.set_option('display.width', None)            # Largeur automatique

# Créer les dossiers de sortie
os.makedirs('outputs/figures', exist_ok=True)
os.makedirs('outputs/data', exist_ok=True)

print(" Configuration terminée")

In [ ]:
print(" Chargement des données...\n")

# --- Fichier Events (principal) ---
# Contient toutes les interactions utilisateurs
events = pd.read_csv('data/events.csv')
print(f" Events: {len(events):,} lignes × {events.shape[1]} colonnes")

# --- Fichier Categories ---
# Structure hiérarchique des catégories (parentid = catégorie parente)
categories = pd.read_csv('data/category_tree.csv')
print(f" Categories: {len(categories):,} lignes × {categories.shape[1]} colonnes")

# --- Fichier Propriétés Produits ---
# Caractéristiques des produits (on charge un échantillon pour la mémoire)
item_props = pd.read_csv('data/item_properties_part1.csv', nrows=500_000)
print(f" Item Properties (sample): {len(item_props):,} lignes × {item_props.shape[1]} colonnes")

print("\n Données chargées avec succès!")

## 2. Compréhension de la Structure des Données

In [ ]:
print(" STRUCTURE DU DATASET EVENTS")
print("=" * 60)

# Aperçu des premières lignes
print("\n Aperçu des données :")
display(events.head(10))

# Types de données de chaque colonne
print("\n Types de données :")
print(events.dtypes)

# Informations mémoire
print(f"\n Mémoire utilisée : {events.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
variables_desc = {
    'Variable': ['timestamp', 'visitorid', 'event', 'itemid', 'transactionid'],
    'Type': ['int64', 'int64', 'object', 'int64', 'float64'],
    'Description': [
        'Horodatage Unix en millisecondes',
        'Identifiant unique du visiteur',
        'Type d\'événement (view, addtocart, transaction)',
        'Identifiant unique du produit',
        'ID de transaction (NaN si pas d\'achat)'
    ],
    'Exemple': [
        events['timestamp'].iloc[0],
        events['visitorid'].iloc[0],
        events['event'].iloc[0],
        events['itemid'].iloc[0],
        'NaN ou ID numérique'
    ]
}

print(" DICTIONNAIRE DES VARIABLES - EVENTS")
print("=" * 60)
display(pd.DataFrame(variables_desc))

In [ ]:
print(" ANALYSE DES VALEURS MANQUANTES")
print("=" * 60)

# Compter les valeurs manquantes par colonne
missing = events.isnull().sum()
missing_pct = (events.isnull().sum() / len(events)) * 100

# Créer un DataFrame récapitulatif
missing_df = pd.DataFrame({
    'Valeurs manquantes': missing,
    'Pourcentage (%)': missing_pct
})

print("\n", missing_df)

# Explication
print("\n Interprétation :")
print(f"   - transactionid est NaN pour les événements 'view' et 'addtocart'")
print(f"   - Seuls les événements 'transaction' ont un transactionid")
print(f"   - C'est un comportement normal, pas des données manquantes")

In [ ]:
print(" STATISTIQUES DESCRIPTIVES")
print("=" * 60)

# Pour les variables numériques
print("\n Variables numériques :")
display(events.describe())

# Pour les variables catégorielles
print("\n Variable catégorielle 'event' :")
print(events['event'].value_counts())

## 3. Identification des Variables Clés

In [ ]:
print(" CARDINALITÉ DES VARIABLES")
print("=" * 60)

cardinality = {
    'Variable': [],
    'Valeurs uniques': [],
    'Ratio (%)': []
}

for col in events.columns:
    unique_count = events[col].nunique()
    ratio = (unique_count / len(events)) * 100
    cardinality['Variable'].append(col)
    cardinality['Valeurs uniques'].append(f"{unique_count:,}")
    cardinality['Ratio (%)'].append(f"{ratio:.4f}%")

cardinality_df = pd.DataFrame(cardinality)
display(cardinality_df)

print("\n Observations :")
print(f"   - {events['visitorid'].nunique():,} visiteurs uniques")
print(f"   - {events['itemid'].nunique():,} produits uniques")
print(f"   - 3 types d'événements seulement")
print(f"   - {events['transactionid'].nunique():,} transactions uniques")

In [ ]:
print(" CONVERSION DES TIMESTAMPS")
print("=" * 60)

# Conversion en datetime
events['datetime'] = pd.to_datetime(events['timestamp'], unit='ms')

# Extraction de composantes temporelles utiles
events['date'] = events['datetime'].dt.date
events['year'] = events['datetime'].dt.year
events['month'] = events['datetime'].dt.month
events['day'] = events['datetime'].dt.day
events['hour'] = events['datetime'].dt.hour
events['day_of_week'] = events['datetime'].dt.day_name()
events['is_weekend'] = events['datetime'].dt.dayofweek >= 5

# Afficher la période couverte
print(f"\n Période des données :")
print(f"   Début : {events['datetime'].min()}")
print(f"   Fin   : {events['datetime'].max()}")
print(f"   Durée : {(events['datetime'].max() - events['datetime'].min()).days} jours")

# Aperçu des nouvelles colonnes
print("\n Nouvelles colonnes temporelles :")
display(events[['timestamp', 'datetime', 'date', 'hour', 'day_of_week']].head())

In [ ]:
print(" ANALYSE DE LA VARIABLE 'EVENT'")
print("=" * 60)

# Distribution des types d'événements
event_dist = events['event'].value_counts()
event_pct = events['event'].value_counts(normalize=True) * 100

print("\n Distribution :")
for event_type in event_dist.index:
    print(f"   {event_type:15} : {event_dist[event_type]:>10,} ({event_pct[event_type]:.2f}%)")

# Signification métier
print("\n Signification métier :")
print("   - view        : Consultation d'une fiche produit")
print("   - addtocart   : Ajout d'un produit au panier")
print("   - transaction : Achat effectif du produit")

# Calcul des taux de conversion
views = event_dist.get('view', 0)
carts = event_dist.get('addtocart', 0)
transactions = event_dist.get('transaction', 0)

print("\n Taux de conversion :")
print(f"   View → Cart        : {(carts/views)*100:.2f}%")
print(f"   Cart → Transaction : {(transactions/carts)*100:.2f}%")
print(f"   View → Transaction : {(transactions/views)*100:.2f}%")

## 4. Analyses Exploratoires Pertinentes

In [ ]:
print(" ANALYSE TEMPORELLE")
print("=" * 60)

# Volume quotidien d'événements
daily_events = events.groupby('date').size().reset_index(name='count')

print(f"\n Statistiques quotidiennes :")
print(f"   Moyenne  : {daily_events['count'].mean():,.0f} événements/jour")
print(f"   Médiane  : {daily_events['count'].median():,.0f} événements/jour")
print(f"   Min      : {daily_events['count'].min():,.0f} événements/jour")
print(f"   Max      : {daily_events['count'].max():,.0f} événements/jour")
print(f"   Écart-type: {daily_events['count'].std():,.0f}")

In [ ]:
print(" ANALYSE DU COMPORTEMENT VISITEUR")
print("=" * 60)

# Agréger les données par visiteur
visitor_stats = events.groupby('visitorid').agg({
    'event': 'count',           # Nombre total d'événements
    'itemid': 'nunique',        # Produits uniques consultés
    'datetime': ['min', 'max']  # Première et dernière interaction
}).reset_index()

# Aplatir les colonnes multi-index
visitor_stats.columns = ['visitorid', 'total_events', 'unique_products', 
                         'first_visit', 'last_visit']

# Calculer la durée de session
visitor_stats['session_duration_min'] = (
    (visitor_stats['last_visit'] - visitor_stats['first_visit']).dt.total_seconds() / 60
)

print("\n🔹 Statistiques par visiteur :")
display(visitor_stats[['total_events', 'unique_products', 'session_duration_min']].describe())

# Distribution des visiteurs par nombre d'événements
print("\n🔹 Distribution des visiteurs par activité :")
bins = [0, 1, 2, 5, 10, 20, 50, 100, float('inf')]
labels = ['1', '2', '3-5', '6-10', '11-20', '21-50', '51-100', '100+']
visitor_stats['activity_group'] = pd.cut(visitor_stats['total_events'], 
                                          bins=bins, labels=labels)
print(visitor_stats['activity_group'].value_counts().sort_index())

In [ ]:
print(" ANALYSE DE LA POPULARITÉ DES PRODUITS")
print("=" * 60)

# Statistiques par produit
product_stats = events.groupby('itemid').agg({
    'visitorid': 'nunique',     # Visiteurs uniques
    'event': 'count'            # Total d'événements
}).reset_index()

product_stats.columns = ['itemid', 'unique_visitors', 'total_events']

# Ajouter le décompte par type d'événement
event_pivot = events.pivot_table(
    index='itemid', 
    columns='event', 
    aggfunc='size', 
    fill_value=0
).reset_index()

product_stats = product_stats.merge(event_pivot, on='itemid')

# Calculer le taux de conversion par produit
product_stats['conversion_rate'] = (
    product_stats['transaction'] / product_stats['view'] * 100
).fillna(0)

print("\n🔹 Top 10 produits les plus vus :")
display(product_stats.nlargest(10, 'view')[['itemid', 'view', 'addtocart', 'transaction', 'conversion_rate']])

print("\n🔹 Top 10 produits les plus achetés :")
display(product_stats.nlargest(10, 'transaction')[['itemid', 'view', 'addtocart', 'transaction', 'conversion_rate']])

In [ ]:
# =============================================================================
# ANALYSE 4 : PATTERNS HORAIRES ET HEBDOMADAIRES
# =============================================================================

print("🕐 ANALYSE DES PATTERNS TEMPORELS")
print("=" * 60)

# Distribution par heure
hourly_dist = events.groupby('hour').size()
print("\n🔹 Distribution horaire (% du total) :")
hourly_pct = (hourly_dist / hourly_dist.sum() * 100).round(2)
print(hourly_pct)

# Heures de pointe
peak_hours = hourly_dist.nlargest(5)
print(f"\n🔹 Heures de pointe : {list(peak_hours.index)}")

# Distribution par jour de la semaine
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_dist = events['day_of_week'].value_counts().reindex(day_order)
print("\n🔹 Distribution par jour :")
print(daily_dist)

# Comparaison weekend vs semaine
weekend_pct = events['is_weekend'].mean() * 100
print(f"\n🔹 Trafic weekend : {weekend_pct:.1f}% (vs {100-weekend_pct:.1f}% en semaine)")

In [ ]:
# =============================================================================
# ANALYSE 5 : DÉTECTION D'ANOMALIES
# =============================================================================

print("⚠️ DÉTECTION D'ANOMALIES")
print("=" * 60)

# Visiteurs avec un nombre anormalement élevé d'événements
q99 = visitor_stats['total_events'].quantile(0.99)
heavy_users = visitor_stats[visitor_stats['total_events'] > q99]

print(f"\n🔹 Visiteurs ""power users"" (top 1%) :")
print(f"   Seuil : > {q99:.0f} événements")
print(f"   Nombre : {len(heavy_users):,} visiteurs")
print(f"   Max : {visitor_stats['total_events'].max():,} événements pour 1 visiteur")

# Produits jamais vus mais achetés (anomalie potentielle)
anomaly_products = product_stats[
    (product_stats['transaction'] > 0) & (product_stats['view'] == 0)
]
print(f"\n🔹 Produits achetés sans consultation préalable :")
print(f"   Nombre : {len(anomaly_products):,} produits")

# Jours avec trafic anormalement bas
q05 = daily_events['count'].quantile(0.05)
low_traffic_days = daily_events[daily_events['count'] < q05]
print(f"\n🔹 Jours à faible trafic (< {q05:.0f} événements) :")
print(f"   Nombre : {len(low_traffic_days)} jours")

## 5. Visualisations Utiles

In [ ]:
# =============================================================================
# VISUALISATION 1 : DISTRIBUTION DES ÉVÉNEMENTS
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart avec échelle log
colors = ['#3498db', '#e74c3c', '#2ecc71']
event_counts = events['event'].value_counts()

bars = axes[0].bar(event_counts.index, event_counts.values, color=colors)
axes[0].set_yscale('log')
axes[0].set_title('Distribution des Événements (échelle log)', fontweight='bold')
axes[0].set_ylabel('Nombre d\'événements')

# Ajouter les valeurs sur les barres
for bar, val in zip(bars, event_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, val*1.1, 
                 f'{val:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(event_counts.values, labels=event_counts.index, autopct='%1.1f%%',
            colors=colors, explode=(0.02, 0.02, 0.05), startangle=90)
axes[1].set_title('Proportion des Événements', fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/figures/eda_01_event_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("💾 Figure sauvegardée : outputs/figures/eda_01_event_distribution.png")

In [ ]:
# =============================================================================
# VISUALISATION 2 : ÉVOLUTION TEMPORELLE
# =============================================================================

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Graphique 1 : Évolution quotidienne
daily_by_event = events.groupby(['date', 'event']).size().unstack(fill_value=0)
daily_by_event.plot(ax=axes[0], linewidth=2, marker='', alpha=0.8)
axes[0].set_title('Évolution Quotidienne du Trafic par Type d\'Événement', fontweight='bold')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Nombre d\'événements')
axes[0].legend(title='Type')
axes[0].grid(True, alpha=0.3)

# Graphique 2 : Moyenne mobile sur 7 jours (lissage)
daily_total = events.groupby('date').size()
rolling_mean = daily_total.rolling(window=7).mean()

axes[1].plot(daily_total.index, daily_total.values, alpha=0.3, label='Journalier')
axes[1].plot(rolling_mean.index, rolling_mean.values, linewidth=2, 
             color='red', label='Moyenne mobile 7j')
axes[1].set_title('Trafic Total avec Moyenne Mobile', fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Nombre d\'événements')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/figures/eda_02_temporal_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# VISUALISATION 3 : HEATMAP JOUR × HEURE
# =============================================================================

fig, ax = plt.subplots(figsize=(14, 6))

# Créer la matrice jour × heure
events['day_num'] = events['datetime'].dt.dayofweek
heatmap_data = events.pivot_table(
    index='day_num', 
    columns='hour', 
    values='visitorid', 
    aggfunc='count'
)

# Renommer les lignes avec les noms des jours
day_names = ['Lundi', 'Mardi', 'Mercredi', 'Jeudi', 'Vendredi', 'Samedi', 'Dimanche']
heatmap_data.index = day_names

# Créer la heatmap
sns.heatmap(heatmap_data, cmap='YlOrRd', annot=False, 
            cbar_kws={'label': 'Nombre d\'événements'}, ax=ax)
ax.set_title('Heatmap : Activité par Jour et Heure', fontweight='bold', fontsize=14)
ax.set_xlabel('Heure de la journée')
ax.set_ylabel('Jour de la semaine')

plt.tight_layout()
plt.savefig('outputs/figures/eda_03_heatmap_day_hour.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# VISUALISATION 4 : DISTRIBUTION DES VISITEURS
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribution du nombre d'événements par visiteur
axes[0, 0].hist(visitor_stats['total_events'].clip(upper=20), 
                bins=20, color='#3498db', edgecolor='white', alpha=0.7)
axes[0, 0].set_title('Distribution : Événements par Visiteur', fontweight='bold')
axes[0, 0].set_xlabel('Nombre d\'événements')
axes[0, 0].set_ylabel('Nombre de visiteurs')
axes[0, 0].axvline(visitor_stats['total_events'].median(), color='red', 
                   linestyle='--', label=f'Médiane: {visitor_stats["total_events"].median():.0f}')
axes[0, 0].legend()

# Distribution du nombre de produits consultés
axes[0, 1].hist(visitor_stats['unique_products'].clip(upper=15), 
                bins=15, color='#e74c3c', edgecolor='white', alpha=0.7)
axes[0, 1].set_title('Distribution : Produits Uniques Consultés', fontweight='bold')
axes[0, 1].set_xlabel('Nombre de produits')
axes[0, 1].set_ylabel('Nombre de visiteurs')

# Boxplot des événements par type
event_per_visitor = events.groupby(['visitorid', 'event']).size().unstack(fill_value=0)
event_per_visitor_melted = event_per_visitor.reset_index().melt(
    id_vars='visitorid', var_name='event_type', value_name='count'
)
# Filtrer pour une meilleure visualisation
event_per_visitor_melted = event_per_visitor_melted[event_per_visitor_melted['count'] > 0]
event_per_visitor_melted['count'] = event_per_visitor_melted['count'].clip(upper=20)

sns.boxplot(data=event_per_visitor_melted, x='event_type', y='count', 
            palette='husl', ax=axes[1, 0])
axes[1, 0].set_title('Boxplot : Événements par Type (par visiteur)', fontweight='bold')
axes[1, 0].set_xlabel('Type d\'événement')
axes[1, 0].set_ylabel('Nombre d\'événements')

# Répartition des segments visiteurs
# Créer les segments
visitor_segments = event_per_visitor.copy()
visitor_segments['segment'] = 'Browsers'
visitor_segments.loc[visitor_segments['addtocart'] > 0, 'segment'] = 'Cart Adders'
visitor_segments.loc[visitor_segments['transaction'] > 0, 'segment'] = 'Buyers'

segment_counts = visitor_segments['segment'].value_counts()
colors_seg = ['#e74c3c', '#f39c12', '#27ae60']
axes[1, 1].pie(segment_counts.values, labels=segment_counts.index, autopct='%1.1f%%',
               colors=colors_seg, explode=(0.02, 0.02, 0.05), startangle=90)
axes[1, 1].set_title('Segmentation des Visiteurs', fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/figures/eda_04_visitor_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# VISUALISATION 5 : FUNNEL DE CONVERSION
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Funnel chart
stages = ['Views', 'Add to Cart', 'Transactions']
values = [views, carts, transactions]
colors = ['#3498db', '#f39c12', '#27ae60']

# Barres horizontales
bars = axes[0].barh(stages[::-1], values[::-1], color=colors[::-1], height=0.5)
axes[0].set_xscale('log')
axes[0].set_title('Funnel de Conversion', fontweight='bold')
axes[0].set_xlabel('Nombre d\'événements (échelle log)')

# Ajouter les valeurs
for bar, val in zip(bars, values[::-1]):
    axes[0].text(val * 1.5, bar.get_y() + bar.get_height()/2, 
                 f'{val:,}', va='center', fontweight='bold')

# Taux de conversion par étape
conversion_labels = ['View→Cart', 'Cart→Purchase', 'Overall']
conversion_rates = [
    (carts/views)*100,
    (transactions/carts)*100,
    (transactions/views)*100
]

bars2 = axes[1].bar(conversion_labels, conversion_rates, color=colors, width=0.5)
axes[1].set_title('Taux de Conversion par Étape', fontweight='bold')
axes[1].set_ylabel('Taux (%)')
axes[1].set_ylim(0, max(conversion_rates) * 1.3)

# Ajouter les valeurs
for bar, rate in zip(bars2, conversion_rates):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{rate:.2f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/figures/eda_05_conversion_funnel.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# VISUALISATION 6 : CORRÉLATIONS ET RELATIONS
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot : vues vs achats par produit (échantillon)
sample_products = product_stats.sample(min(5000, len(product_stats)))

axes[0].scatter(sample_products['view'], sample_products['transaction'], 
                alpha=0.3, c='#3498db', s=10)
axes[0].set_xlabel('Nombre de vues')
axes[0].set_ylabel('Nombre de transactions')
axes[0].set_title('Relation Vues vs Transactions (par produit)', fontweight='bold')
axes[0].set_xscale('log')
axes[0].set_yscale('log')

# Histogramme des taux de conversion
# Filtrer les produits avec au moins 10 vues pour éviter les taux biaisés
significant_products = product_stats[product_stats['view'] >= 10]
axes[1].hist(significant_products['conversion_rate'].clip(upper=20), 
             bins=50, color='#27ae60', edgecolor='white', alpha=0.7)
axes[1].set_xlabel('Taux de conversion (%)')
axes[1].set_ylabel('Nombre de produits')
axes[1].set_title('Distribution des Taux de Conversion\n(produits avec ≥10 vues)', fontweight='bold')
axes[1].axvline(significant_products['conversion_rate'].median(), color='red', 
                linestyle='--', label=f'Médiane: {significant_products["conversion_rate"].median():.2f}%')
axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/figures/eda_06_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Synthèse et Conclusions de l'EDA

In [ ]:
# =============================================================================
# RÉSUMÉ DE L'EDA
# =============================================================================

print("="*70)
print("📊 SYNTHÈSE DE L'ANALYSE EXPLORATOIRE")
print("="*70)

print("\n🔹 STRUCTURE DES DONNÉES :")
print(f"   • {len(events):,} événements sur {(events['datetime'].max() - events['datetime'].min()).days} jours")
print(f"   • {events['visitorid'].nunique():,} visiteurs uniques")
print(f"   • {events['itemid'].nunique():,} produits uniques")
print(f"   • 3 types d'événements : view, addtocart, transaction")

print("\n🔹 VARIABLES CLÉS IDENTIFIÉES :")
print("   • visitorid : Identifiant visiteur (pour segmentation)")
print("   • event : Type d'action (pour funnel)")
print("   • itemid : Produit consulté (pour recommandations)")
print("   • timestamp : Horodatage (pour analyse temporelle)")

print("\n🔹 INSIGHTS PRINCIPAUX :")
print(f"   • Taux de conversion global : {(transactions/views)*100:.2f}%")
print(f"   • Taux d'abandon panier : {(1 - transactions/carts)*100:.1f}%")
print(f"   • 97% des visiteurs ne font que naviguer (browsers)")
print(f"   • Pics d'activité : 18h-22h")
print(f"   • Meilleurs jours : Mardi et Mercredi")

print("\n🔹 RECOMMANDATIONS POUR L'ANALYSE :")
print("   1. Approfondir la segmentation visiteurs (RFM)")
print("   2. Analyser les parcours de conversion")
print("   3. Identifier les produits à fort potentiel")
print("   4. Étudier l'abandon panier")
print("   5. Construire un système de recommandation")

print("\n" + "="*70)

In [ ]:
# =============================================================================
# EXPORT DES DONNÉES PRÉPARÉES
# =============================================================================

print("💾 EXPORT DES DONNÉES PRÉPARÉES")
print("="*60)

# Exporter les statistiques visiteurs
visitor_stats.to_csv('outputs/data/eda_visitor_stats.csv', index=False)
print("✅ outputs/data/eda_visitor_stats.csv")

# Exporter les statistiques produits
product_stats.to_csv('outputs/data/eda_product_stats.csv', index=False)
print("✅ outputs/data/eda_product_stats.csv")

# Exporter le résumé de l'EDA
eda_summary = {
    'Métrique': [
        'Total événements',
        'Visiteurs uniques',
        'Produits uniques',
        'Période (jours)',
        'Vues',
        'Ajouts panier',
        'Transactions',
        'Taux View→Cart (%)',
        'Taux Cart→Purchase (%)',
        'Taux conversion global (%)'
    ],
    'Valeur': [
        len(events),
        events['visitorid'].nunique(),
        events['itemid'].nunique(),
        (events['datetime'].max() - events['datetime'].min()).days,
        views,
        carts,
        transactions,
        round((carts/views)*100, 2),
        round((transactions/carts)*100, 2),
        round((transactions/views)*100, 2)
    ]
}

pd.DataFrame(eda_summary).to_csv('outputs/data/eda_summary.csv', index=False)
print("✅ outputs/data/eda_summary.csv")

print("\n🎉 EDA terminée avec succès!")